# Batch Processing

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SPARK_API").getOrCreate()

In [ ]:
from pyspark.sql.functions import count

df = spark.read.option("header", "true").option("inferSchema", "true").csv("data/titanic.csv")

passenger_counts = df.groupBy("Pclass").agg(count("*").alias("count"))

passenger_counts.show()


+------+-----+
|Pclass|count|
+------+-----+
|     1|  216|
|     3|  491|
|     2|  184|
+------+-----+



In [ ]:
from pyspark.sql.functions import when

age_ranges = [
    (0, 10), (10, 20), (20, 30), (30, 40),
    (40, 50), (50, 60), (60, 70), (70, 80)
]

age_counts = []

for lower, upper in age_ranges:
    if lower == 0:
        condition = (df.Age >= lower) & (df.Age <= upper)
        label = "0-10"
    else:
        condition = (df.Age > lower) & (df.Age <= upper)
        label = f">{lower}-{upper}"

    age_counts.append(
        count(when(condition, True)).alias(label)
    )

passenger_counts_by_age = (
    df.groupBy("Pclass")
      .agg(*age_counts)
      .orderBy("Pclass")
)

passenger_counts_by_age.show()

+------+----+------+------+------+------+------+------+------+
|Pclass|0-10|>10-20|>20-30|>30-40|>40-50|>50-60|>60-70|>70-80|
+------+----+------+------+------+------+------+------+------+
|     1|   3|    18|    40|    49|    37|    25|    11|     3|
|     2|  17|    18|    61|    43|    19|    12|     3|     0|
|     3|  44|    79|   129|    63|    30|     5|     3|     2|
+------+----+------+------+------+------+------+------+------+



In [ ]:
passenger_counts_by_sex = (df.groupBy("Pclass", "Sex").agg(count("*").alias("count"))
)

passenger_counts_by_sex.show()


+------+------+-----+
|Pclass|   Sex|count|
+------+------+-----+
|     2|female|   76|
|     3|  male|  347|
|     1|  male|  122|
|     3|female|  144|
|     1|female|   94|
|     2|  male|  108|
+------+------+-----+



In [ ]:
survivor_counts = df.groupBy("Survived").agg(count("*").alias("count"))

survivor_counts.show()


+--------+-----+
|Survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



In [ ]:
from pyspark.sql.functions import avg

avg_age_by_sex_and_class = (
    df.groupBy("Pclass", "Sex")
      .agg(avg("Age").alias("avg_age"))
)

avg_age_by_sex_and_class.show()


+------+------+------------------+
|Pclass|   Sex|           avg_age|
+------+------+------------------+
|     2|female|28.722972972972972|
|     3|  male|26.507588932806325|
|     1|  male| 41.28138613861386|
|     3|female|             21.75|
|     1|female| 34.61176470588235|
|     2|  male| 30.74070707070707|
+------+------+------------------+



In [ ]:
avg_age_first_class = (
    df.filter(df.Pclass == 1)
      .agg(avg("Age").alias("avg_age"))
)

avg_age_first_class.show()


+------------------+
|           avg_age|
+------------------+
|38.233440860215055|
+------------------+



In [ ]:
avg_age_by_class_and_survival = (
    df.groupBy("Pclass", "Survived")
      .agg(avg("Age").alias("avg_age"))
)

avg_age_by_class_and_survival.show()


+------+--------+------------------+
|Pclass|Survived|           avg_age|
+------+--------+------------------+
|     1|       0|        43.6953125|
|     3|       1|20.646117647058823|
|     1|       1| 35.36819672131148|
|     2|       1| 25.90156626506024|
|     2|       0|33.544444444444444|
|     3|       0|26.555555555555557|
+------+--------+------------------+



In [ ]:
from pyspark.sql.functions import min, max

age_min_max = df.agg(
    min("Age").alias("min_age"),
    max("Age").alias("max_age")
)

age_min_max.show()



+-------+-------+
|min_age|max_age|
+-------+-------+
|   0.42|   80.0|
+-------+-------+



In [ ]:
from pyspark.sql.functions import col

passenger_counts_first_class_30_plus = (
    df.filter((col("Pclass") == 1) & (col("Age") >= 30))
      .groupBy("Sex")
      .agg(count("*").alias("count"))
)

passenger_counts_first_class_30_plus.show()


+------+-----+
|   Sex|count|
+------+-----+
|female|   55|
|  male|   76|
+------+-----+



In [ ]:
survival_by_cabin = (
    df.filter(df.Cabin.isNotNull())
      .groupBy("Cabin")
      .agg(avg("Survived").alias("survival_rate"))
      .orderBy("Cabin")
)

survival_by_cabin.show(20, truncate=False)



+-----+-------------+
|Cabin|survival_rate|
+-----+-------------+
|A10  |0.0          |
|A14  |0.0          |
|A16  |1.0          |
|A19  |0.0          |
|A20  |1.0          |
|A23  |1.0          |
|A24  |0.0          |
|A26  |1.0          |
|A31  |1.0          |
|A32  |0.0          |
|A34  |1.0          |
|A36  |0.0          |
|A5   |0.0          |
|A6   |1.0          |
|A7   |0.0          |
|B101 |1.0          |
|B102 |0.0          |
|B18  |1.0          |
|B19  |0.0          |
|B20  |1.0          |
+-----+-------------+
only showing top 20 rows


In [ ]:
survival_by_embarked = (
    df.groupBy("Embarked")
      .agg(avg("Survived").alias("survival_rate"))
      .orderBy("Embarked")
)

survival_by_embarked.show()


+--------+-------------------+
|Embarked|      survival_rate|
+--------+-------------------+
|    NULL|                1.0|
|       C| 0.5535714285714286|
|       Q|0.38961038961038963|
|       S|0.33695652173913043|
+--------+-------------------+



In [ ]:
from pyspark.sql.functions import round

survival_by_rounded_age = (
    df.withColumn("Age_rounded", round(col("Age") / 10) * 10)
      .groupBy("Age_rounded")
      .agg(
          count("*").alias("count"),
          avg("Survived").alias("avg_survived")
      )
      .orderBy(col("count").desc())
)

survival_by_rounded_age.show()

+-----------+-----+-------------------+
|Age_rounded|count|       avg_survived|
+-----------+-----+-------------------+
|       30.0|  201| 0.3880597014925373|
|       20.0|  200|              0.365|
|       NULL|  177| 0.2937853107344633|
|       40.0|  120|              0.425|
|       50.0|   73|  0.410958904109589|
|        0.0|   40|              0.675|
|       10.0|   38|0.47368421052631576|
|       60.0|   31| 0.3870967741935484|
|       70.0|   10|                0.0|
|       80.0|    1|                1.0|
+-----------+-----+-------------------+



In [ ]:
avg_age_and_class_by_survival = (
    df.groupBy("Survived")
      .agg(
          avg("Age").alias("avg_age"),
          avg("Pclass").alias("avg_pclass")
      )
)

avg_age_and_class_by_survival.show()


+--------+------------------+------------------+
|Survived|           avg_age|        avg_pclass|
+--------+------------------+------------------+
|       1|28.343689655172415|1.9502923976608186|
|       0| 30.62617924528302|2.5318761384335153|
+--------+------------------+------------------+



In [ ]:
survival_without_cabin = (
    df.filter(col("Cabin").isNull())
      .agg(avg("Survived").alias("survival_rate"))
)

survival_without_cabin.show()

+-------------------+
|      survival_rate|
+-------------------+
|0.29985443959243085|
+-------------------+



In [ ]:
passenger_counts_without_cabin = (
    df.filter(col("Cabin").isNull())
      .groupBy("Pclass", "Survived")
      .agg(count("*").alias("count"))
)

passenger_counts_without_cabin.show()

+------+--------+-----+
|Pclass|Survived|count|
+------+--------+-----+
|     1|       0|   21|
|     3|       1|  113|
|     1|       1|   19|
|     2|       1|   74|
|     2|       0|   94|
|     3|       0|  366|
+------+--------+-----+

